# Notebook 07 — Input Ablation + Leave-One-Task-Out (LOTO) Generalization

## Goals
This notebook contains two experiment blocks that strengthen the experimental protocol:

### (A) Input-field ablation (fair comparison)
Run the **same** TF-IDF + Logistic Regression pipeline on different input compositions:
- `response_only`
- `prompt_only`
- `prompt_plus_response`
- `response_plus_context`
- `prompt_plus_response_plus_context`

We report metrics on **Validation** and **Test** for each setting.

### (B) Leave-One-Task-Out generalization (LOTO)
For each task in {qa, dialogue, summarization, general}:
- Train on samples from the other 3 tasks (train split only)
- Evaluate on the **held-out task** in Val and Test

This tests whether performance is driven by task-specific artifacts or generalizable signals.

## Outputs
We save experiment tables as CSV files under:
- `reports/nb07_input_ablation.csv`
- `reports/nb07_loto.csv`

In [1]:
# Core
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Metrics
from sklearn.metrics import roc_auc_score

# Project utils (keep consistent with your repo)
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load_splits import load_splits
from src.utils.eval import evaluate_split

# Repro
SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_colwidth", 200)

# Output dir
REPORTS_DIR = ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("REPORTS_DIR:", REPORTS_DIR)

ROOT: /Users/aviv.gross/hallu-detect
REPORTS_DIR: /Users/aviv.gross/hallu-detect/reports


## Load processed splits 
We load `train/val/test` using `load_splits` to ensure we use the latest processed files and consistent schema.

In [2]:
train_df, val_df, test_df = load_splits(root=ROOT)

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

required = {"prompt", "response", "context", "label", "task", "group_id"}
missing_train = required - set(train_df.columns)
missing_val   = required - set(val_df.columns)
missing_test  = required - set(test_df.columns)
print("Missing cols (train/val/test):", missing_train, missing_val, missing_test)

print("\nLabel mean (train/val/test):",
      float(train_df["label"].mean()),
      float(val_df["label"].mean()),
      float(test_df["label"].mean()))

print("\nTask counts (train):")
display(train_df["task"].value_counts())

Train: (51647, 7)
Val:   (6425, 7)
Test:  (6435, 7)
Missing cols (train/val/test): set() set() set()

Label mean (train/val/test): 0.4655062249501423 0.4628793774319066 0.4637140637140637

Task counts (train):


task
summarization    16054
dialogue         16020
qa               16010
general           3563
Name: count, dtype: int64

## Define input composition ("text views")
We create a single text column based on a chosen setting.

Important:
- We keep **the same model hyperparameters** across settings to ensure a fair comparison.
- We intentionally start with text-only models here (numeric features are handled in a separate ablation notebook).

In [4]:
def build_text(df: pd.DataFrame, setting: str) -> pd.Series:
    """
    Build a single text field from different source columns.
    This is the ONLY thing that changes across ablation settings.
    """
    prompt = df["prompt"].fillna("").astype(str)
    response = df["response"].fillna("").astype(str)
    context = df["context"].fillna("").astype(str)

    if setting == "response_only":
        return response

    if setting == "prompt_only":
        return prompt

    if setting == "prompt_plus_response":
        return prompt + "\n\n" + response

    if setting == "response_plus_context":
        # put context first to bias TF-IDF towards “supporting knowledge”
        return context + "\n\n" + response

    if setting == "prompt_plus_response_plus_context":
        return prompt + "\n\n" + context + "\n\n" + response

    raise ValueError(f"Unknown setting: {setting}")


def make_tfidf_lr_pipeline(
    *,
    max_features: int = 100_000,
    ngram_range=(1, 2),
    min_df: int = 2,
    C: float = 1.0,
    class_weight="balanced",
    seed: int = SEED,
) -> Pipeline:
    """
    TF-IDF + Logistic Regression pipeline.
    Keep this identical across settings for fair ablation.
    """
    return Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(
                max_features=max_features,
                ngram_range=ngram_range,
                min_df=min_df,
                strip_accents=None,
                lowercase=True
            )),
            ("lr", LogisticRegression(
                C=C,
                max_iter=2000,
                class_weight=class_weight,
                random_state=seed,
                n_jobs=None  # keep silent; sklearn ignores it in newer versions anyway
            ))
        ]
    )


# Default hyperparams (match your baseline spirit; easy to tweak here)
TFIDF_PARAMS = dict(
    max_features=100_000,
    ngram_range=(1, 2),
    min_df=2,
    C=1.0,
    class_weight="balanced",
    seed=SEED
)

print("TFIDF_PARAMS:", TFIDF_PARAMS)

TFIDF_PARAMS: {'max_features': 100000, 'ngram_range': (1, 2), 'min_df': 2, 'C': 1.0, 'class_weight': 'balanced', 'seed': 42}


## (A) Input-field ablation
We run the same pipeline on multiple input settings and collect:
- Validation metrics
- Test metrics
- ROC-AUC (Val/Test) using predicted probabilities

Outputs are saved to `reports/nb07_input_ablation.csv`.

In [7]:
from sklearn.metrics import roc_auc_score

def _metrics_to_dict(m):
    """
    Convert SplitMetrics (dataclass / object) or dict into a plain dict.
    Compatible with your src.utils.eval.evaluate_split return type.
    """
    if isinstance(m, dict):
        return dict(m)

    # dataclass case
    try:
        import dataclasses
        if dataclasses.is_dataclass(m):
            return dataclasses.asdict(m)
    except Exception:
        pass

    # generic object: take common public attributes if present
    d = {}
    for k in ["accuracy", "f1", "precision", "recall", "confusion_matrix"]:
        if hasattr(m, k):
            d[k] = getattr(m, k)
    return d


def eval_with_auc(split_name: str, model, X, y) -> dict:
    """
    Wrap evaluate_split(...) and return a dict with an additional 'roc_auc' field.
    """
    m_obj = evaluate_split(split_name, model, X, y)
    metrics = _metrics_to_dict(m_obj)

    auc = None
    if hasattr(model, "predict_proba"):
        try:
            y_prob = model.predict_proba(X)[:, 1]
            auc = float(roc_auc_score(y, y_prob))
        except Exception:
            auc = None

    metrics["roc_auc"] = auc
    return metrics


def run_input_ablation(settings: list[str]) -> pd.DataFrame:
    rows = []

    for setting in settings:
        print("\n" + "="*90)
        print(f"SETTING: {setting}")
        print("="*90)

        X_train = build_text(train_df, setting)
        y_train = train_df["label"].values

        X_val = build_text(val_df, setting)
        y_val = val_df["label"].values

        X_test = build_text(test_df, setting)
        y_test = test_df["label"].values

        pipe = make_tfidf_lr_pipeline(**TFIDF_PARAMS)
        pipe.fit(X_train, y_train)

        val_m = eval_with_auc("val", pipe, X_val, y_val)
        test_m = eval_with_auc("test", pipe, X_test, y_test)

        rows.append({
            "setting": setting,
            "val_accuracy": val_m.get("accuracy"),
            "val_f1": val_m.get("f1"),
            "val_precision": val_m.get("precision"),
            "val_recall": val_m.get("recall"),
            "val_roc_auc": val_m.get("roc_auc"),
            "test_accuracy": test_m.get("accuracy"),
            "test_f1": test_m.get("f1"),
            "test_precision": test_m.get("precision"),
            "test_recall": test_m.get("recall"),
            "test_roc_auc": test_m.get("roc_auc"),
        })

    out = pd.DataFrame(rows).sort_values(by="val_f1", ascending=False).reset_index(drop=True)
    return out


INPUT_SETTINGS = [
    "response_only",
    "prompt_only",
    "prompt_plus_response",
    "response_plus_context",
    "prompt_plus_response_plus_context",
]

ablation_df = run_input_ablation(INPUT_SETTINGS)
display(ablation_df)

ablation_path = REPORTS_DIR / "nb07_input_ablation.csv"
ablation_df.to_csv(ablation_path, index=False)
print("Saved:", ablation_path)


SETTING: response_only

val metrics:
  accuracy = 0.8280
  f1       = 0.8134
  precision= 0.8171
  recall   = 0.8097
  confusion matrix:
[[2912  539]
 [ 566 2408]]

test metrics:
  accuracy = 0.8244
  f1       = 0.8114
  precision= 0.8082
  recall   = 0.8147
  confusion matrix:
[[2874  577]
 [ 553 2431]]

SETTING: prompt_only

val metrics:
  accuracy = 0.5370
  f1       = 0.6121
  precision= 0.4999
  recall   = 0.7892
  confusion matrix:
[[1103 2348]
 [ 627 2347]]

test metrics:
  accuracy = 0.5360
  f1       = 0.6125
  precision= 0.4998
  recall   = 0.7909
  confusion matrix:
[[1089 2362]
 [ 624 2360]]

SETTING: prompt_plus_response

val metrics:
  accuracy = 0.6875
  f1       = 0.6696
  precision= 0.6556
  recall   = 0.6843
  confusion matrix:
[[2382 1069]
 [ 939 2035]]

test metrics:
  accuracy = 0.6796
  f1       = 0.6624
  precision= 0.6476
  recall   = 0.6779
  confusion matrix:
[[2350 1101]
 [ 961 2023]]

SETTING: response_plus_context

val metrics:
  accuracy = 0.7538
  f1    

,setting,val_accuracy,val_f1,val_precision,val_recall,val_roc_auc,test_accuracy,test_f1,test_precision,test_recall,test_roc_auc
0,response_only,0.828016,0.813376,0.817102,0.809684,0.902781,0.824398,0.811415,0.808178,0.814678,0.900248
1,response_plus_context,0.753774,0.743764,0.717500,0.772024,0.843958,0.746542,0.735870,0.712003,0.761394,0.838257
2,prompt_plus_response,0.687471,0.669628,0.655606,0.684264,0.759952,0.679565,0.662410,0.647567,0.677949,0.754226
3,prompt_plus_response_plus_context,0.655720,0.647434,0.615455,0.682919,0.722758,0.652370,0.642366,0.614185,0.673257,0.718543
4,prompt_only,0.536965,0.612075,0.499894,0.789173,0.568949,0.535975,0.612510,0.499788,0.790885,0.567359


Saved: /Users/aviv.gross/hallu-detect/reports/nb07_input_ablation.csv


## Select the best input setting
We will use the best-performing setting on Validation (by F1) as the default for the LOTO generalization block.

In [8]:
best_setting = ablation_df.loc[0, "setting"]
print("Best setting by VAL F1:", best_setting)

# Optional: show the top-3
display(ablation_df.head(3))

Best setting by VAL F1: response_only


,setting,val_accuracy,val_f1,val_precision,val_recall,val_roc_auc,test_accuracy,test_f1,test_precision,test_recall,test_roc_auc
0,response_only,0.828016,0.813376,0.817102,0.809684,0.902781,0.824398,0.811415,0.808178,0.814678,0.900248
1,response_plus_context,0.753774,0.743764,0.717500,0.772024,0.843958,0.746542,0.735870,0.712003,0.761394,0.838257
2,prompt_plus_response,0.687471,0.669628,0.655606,0.684264,0.759952,0.679565,0.662410,0.647567,0.677949,0.754226


## (B) Leave-One-Task-Out (LOTO) Generalization

Protocol:
- For each `heldout_task`:
  - Train using **train split** samples from all other tasks
  - Evaluate on **val split** filtered to `heldout_task`
  - Evaluate on **test split** filtered to `heldout_task`

We use the **best input setting** from the ablation block to avoid cherry-picking across tasks.

Outputs are saved to `reports/nb07_loto.csv`.

In [12]:
def has_two_classes(df: pd.DataFrame) -> bool:
    return df["label"].nunique() >= 2

valid_tasks = []
for t in TASKS:
    v = val_df[val_df["task"] == t]
    te = test_df[test_df["task"] == t]
    if has_two_classes(v) and has_two_classes(te):
        valid_tasks.append(t)
    else:
        print(f"[SKIP] task={t} has <2 classes in val/test "
              f"(val uniques={v['label'].unique()}, test uniques={te['label'].unique()})")

print("Tasks used for LOTO:", valid_tasks)

[SKIP] task=general has <2 classes in val/test (val uniques=[0], test uniques=[0])
Tasks used for LOTO: ['qa', 'dialogue', 'summarization']


In [13]:
TASKS = ["qa", "dialogue", "summarization", "general"]

def run_loto(heldout_tasks: list[str], setting: str) -> pd.DataFrame:
    rows = []

    for heldout in heldout_tasks:
        print("\n" + "#"*90)
        print(f"HELD-OUT TASK: {heldout}")
        print("#"*90)

        train_sub = train_df[train_df["task"] != heldout].copy()
        val_sub   = val_df[val_df["task"] == heldout].copy()
        test_sub  = test_df[test_df["task"] == heldout].copy()

        print("Train subset:", train_sub.shape, "| Val subset:", val_sub.shape, "| Test subset:", test_sub.shape)

        # Guard against tiny subsets (should not happen, but keep notebook robust)
        if len(val_sub) < 50 or len(test_sub) < 50:
            print(f"[WARN] Very small subset for task={heldout}. Metrics may be noisy.")

        X_train = build_text(train_sub, setting)
        y_train = train_sub["label"].values

        X_val = build_text(val_sub, setting)
        y_val = val_sub["label"].values

        X_test = build_text(test_sub, setting)
        y_test = test_sub["label"].values

        pipe = make_tfidf_lr_pipeline(**TFIDF_PARAMS)
        pipe.fit(X_train, y_train)

        val_m = eval_with_auc(f"val:{heldout}", pipe, X_val, y_val)
        test_m = eval_with_auc(f"test:{heldout}", pipe, X_test, y_test)

        rows.append({
            "heldout_task": heldout,
            "setting": setting,
            "train_size": len(train_sub),
            "val_size": len(val_sub),
            "test_size": len(test_sub),

            "val_accuracy": val_m.get("accuracy"),
            "val_f1": val_m.get("f1"),
            "val_precision": val_m.get("precision"),
            "val_recall": val_m.get("recall"),
            "val_roc_auc": val_m.get("roc_auc"),

            "test_accuracy": test_m.get("accuracy"),
            "test_f1": test_m.get("f1"),
            "test_precision": test_m.get("precision"),
            "test_recall": test_m.get("recall"),
            "test_roc_auc": test_m.get("roc_auc"),
        })

    out = pd.DataFrame(rows).sort_values(by="test_f1", ascending=False).reset_index(drop=True)
    return out


loto_df = run_loto(valid_tasks, setting=best_setting)
display(loto_df)

loto_path = REPORTS_DIR / "nb07_loto.csv"
loto_df.to_csv(loto_path, index=False)
print("Saved:", loto_path)


##########################################################################################
HELD-OUT TASK: qa
##########################################################################################
Train subset: (35637, 7) | Val subset: (2008, 7) | Test subset: (1982, 7)

val:qa metrics:
  accuracy = 0.6106
  f1       = 0.4164
  precision= 0.8304
  recall   = 0.2779
  confusion matrix:
[[947  57]
 [725 279]]

test:qa metrics:
  accuracy = 0.5949
  f1       = 0.3847
  precision= 0.7994
  recall   = 0.2533
  confusion matrix:
[[928  63]
 [740 251]]

##########################################################################################
HELD-OUT TASK: dialogue
##########################################################################################
Train subset: (35627, 7) | Val subset: (1938, 7) | Test subset: (2042, 7)

val:dialogue metrics:
  accuracy = 0.6352
  f1       = 0.6427
  precision= 0.6297
  recall   = 0.6563
  confusion matrix:
[[595 374]
 [333 636]]

test:dialogue me

,heldout_task,setting,train_size,val_size,test_size,val_accuracy,val_f1,val_precision,val_recall,val_roc_auc,test_accuracy,test_f1,test_precision,test_recall,test_roc_auc
0,dialogue,response_only,35627,1938,2042,0.635191,0.642749,0.629703,0.656347,0.686485,0.626347,0.627623,0.625486,0.629775,0.677150
1,summarization,response_only,35593,2002,1944,0.516484,0.598673,0.511694,0.721279,0.545028,0.535494,0.614925,0.525127,0.741770,0.548639
2,qa,response_only,35637,2008,1982,0.610558,0.416418,0.830357,0.277888,0.788344,0.594854,0.384674,0.799363,0.253280,0.789792


Saved: /Users/aviv.gross/hallu-detect/reports/nb07_loto.csv


## Summary (for writing later)
This cell prints a short, copy-paste friendly summary of the key findings:
- Best input setting from ablation
- Best / worst LOTO task performance
- Average LOTO performance

In [15]:
print("=== Input ablation ===")
print("Best setting:", best_setting)
print("Top-3 settings by VAL F1:")
display(ablation_df[["setting", "val_f1", "val_accuracy", "val_roc_auc", "test_f1", "test_accuracy", "test_roc_auc"]].head(3))

print("\n=== LOTO summary ===")
avg_test_f1 = float(loto_df["test_f1"].mean())
min_row = loto_df.loc[loto_df["test_f1"].idxmin()]
max_row = loto_df.loc[loto_df["test_f1"].idxmax()]

print("Average test F1 across held-out tasks:", round(avg_test_f1, 4))
print("Best held-out task:", max_row["heldout_task"], "| test_f1=", round(float(max_row["test_f1"]), 4))
print("Worst held-out task:", min_row["heldout_task"], "| test_f1=", round(float(min_row["test_f1"]), 4))

display(loto_df[["heldout_task", "test_f1", "test_accuracy", "test_roc_auc", "test_precision", "test_recall", "test_size"]])

=== Input ablation ===
Best setting: response_only
Top-3 settings by VAL F1:


,setting,val_f1,val_accuracy,val_roc_auc,test_f1,test_accuracy,test_roc_auc
0,response_only,0.813376,0.828016,0.902781,0.811415,0.824398,0.900248
1,response_plus_context,0.743764,0.753774,0.843958,0.735870,0.746542,0.838257
2,prompt_plus_response,0.669628,0.687471,0.759952,0.662410,0.679565,0.754226



=== LOTO summary ===
Average test F1 across held-out tasks: 0.5424
Best held-out task: dialogue | test_f1= 0.6276
Worst held-out task: qa | test_f1= 0.3847


,heldout_task,test_f1,test_accuracy,test_roc_auc,test_precision,test_recall,test_size
0,dialogue,0.627623,0.626347,0.677150,0.625486,0.629775,2042
1,summarization,0.614925,0.535494,0.548639,0.525127,0.741770,1944
2,qa,0.384674,0.594854,0.789792,0.799363,0.253280,1982
